In [11]:
import requests
from bs4 import BeautifulSoup
import re

In [12]:
url='https://rollcall.com/factbase/trump/transcript/donald-trump-speech-campaign-rally-martinsburg-pennsylvania-october-26-2020'
resp=requests.get(url)
soup=BeautifulSoup(resp.text,'html.parser')
print(soup)

<!DOCTYPE html>

<html lang="en-US">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<link href="https://rollcall.com/app/themes/rollcall/public/favicon.dfdc32.ico" rel="icon" type="image/x-icon"/>
<meta content="#2F3B4B" name="msapplication-TileColor"/>
<link href="https://rollcall.com/app/themes/rollcall/public/pwa-manifest.json" rel="manifest"/>
<meta content="index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1" name="robots">
<style>img:is([sizes="auto" i], [sizes^="auto," i]) { contain-intrinsic-size: 3000px 1500px }</style>
<meta content="yes" name="mobile-web-app-capable"/>
<meta content="yes" name="apple-mobile-web-app-capable"/>
<meta content="black" name="apple-mobile-web-app-status-bar-style"/>
<meta content="#2F3B4B" name="theme-color"/>
<meta content="#2F3B4B" name="msapplication-TileColor"/>
<meta content="https://rollcall.com/app/themes/rollcall/app/uploads/2022/02/cropped-rollcall-placeholder-

Title

In [13]:
speech_title=soup.find(class_="text-[#2F3C4B] text-center text-xl sm:text-2xl not-italic font-semibold leading-normal sm:leading-9 font-graphik")
speech_titles=speech_title.get_text(strip=True)
speech_title=re.search(r"^(.+)-",speech_titles).group(1).strip()
print(speech_titles)
print(speech_title)
speech_date=re.search(r"-\s(.+)",speech_titles).group(1).strip()
speech_date


Speech: Donald Trump Holds a Campaign Rally in Martinsburg, Pennsylvania - October 26, 2020
Speech: Donald Trump Holds a Campaign Rally in Martinsburg, Pennsylvania


'October 26, 2020'

Number of sentences and number of words, all in one, strong

In [14]:
blocks_container=soup.find(class_="flex flex-wrap gap-8 justify-between")
blocks=blocks_container.find_all(class_='flex-1 h-content')
for block in blocks:
    name_div=block.find(class_="font-graphik text-sm font-medium leading-normal flex items-center").get_text(strip=True)
    if "Trump" in name_div:
        trump_block=block
    
contents=trump_block.find_all(class_="font-graphik text-xs font-medium text-[#2F3C4B]")
contents=[content.get_text(strip=True) for content in contents]
contents
for content in contents :
    if "sentences" in content:
        nbr_sentences=re.search(r'\d+',content).group()
    elif "words" in content:
        nbr_words=re.search(r'\d+',content).group()
    else :
        nbr_seconds=re.search(r'\d+',content).group()
print(nbr_sentences,nbr_words,nbr_seconds)

1044 10182 3741


Categories

In [21]:
categories=soup.find_all(class_="text-[#015582] text-sm font-normal leading-normal rounded-md bg-[#F4F4F5] border border-[#D9D9D9] p-2")
categories=[category.get_text(strip=True) for category in categories]
categories



['Politics > Election',
 'Politics > Human Rights',
 'Politics > Politics (General)',
 'Crime, Law and Justice > Justice and Rights']

Cleaning Categories

In [22]:
categories=[category.split('>') for category in categories]
categories=list(set([category[i].strip() for category in categories for i in range(len(category))]))
categories

['Human Rights',
 'Crime, Law and Justice',
 'Election',
 'Justice and Rights',
 'Politics',
 'Politics (General)']

Transcriptions

In [23]:
transcriptions=soup.find_all(class_='flex gap-4 py-2')
list_transcriptions=[]
for transcription in transcriptions :
    speaker=transcription.find(class_="text-md inline").get_text(strip=True)
    timestamp=transcription.find(class_='text-xs text-gray-600 inline ml-2').get_text(strip=True)
    text=transcription.find(class_='flex-auto text-md text-gray-600 leading-loose').get_text(strip=True)
    list_transcriptions.append([speaker,timestamp,text])
list_transcriptions

[['Donald Trump',
  '00:00:00-00:00:28 (28 sec)',
  "Thank you very much, hello. Oh wow. We tried to come in here very low key. We wanted to make a low key appearance. Did anybody notice the helicopter flying a little low and real slow? Greatest pilots in the world. Greatest equipment in the world. That's what we have. Greatest equipment in the world."],
 ['Donald Trump',
  '00:00:28-00:00:50 (22 sec)',
  "But the greatest pilots. Nobody can do what our pilots can do, right, Mike? Hello Pennsylvania. Hello Pennsylvania. Eight days from now. Can you believe it? Eight days. Started off four years ago, remember our meetings and our -- it was a love fest, right from the beginning, wasn't it, and now it's more so."],
 ['Donald Trump',
  '00:00:50-00:01:27 (37 sec)',
  'You know what? It\'s more so than it was four years ago, and they\'re going to be finding that out very soon. They\'re going to be finding it out. You know, I have, I have the ratings right over here. Whole numbers, thank you

trump_transcriptions

In [24]:
trump_transcriptions=[transcription_list[1:] for transcription_list in list_transcriptions if "Donald Trump" in transcription_list[0]]
trump_transcriptions


[['00:00:00-00:00:28 (28 sec)',
  "Thank you very much, hello. Oh wow. We tried to come in here very low key. We wanted to make a low key appearance. Did anybody notice the helicopter flying a little low and real slow? Greatest pilots in the world. Greatest equipment in the world. That's what we have. Greatest equipment in the world."],
 ['00:00:28-00:00:50 (22 sec)',
  "But the greatest pilots. Nobody can do what our pilots can do, right, Mike? Hello Pennsylvania. Hello Pennsylvania. Eight days from now. Can you believe it? Eight days. Started off four years ago, remember our meetings and our -- it was a love fest, right from the beginning, wasn't it, and now it's more so."],
 ['00:00:50-00:01:27 (37 sec)',
  'You know what? It\'s more so than it was four years ago, and they\'re going to be finding that out very soon. They\'re going to be finding it out. You know, I have, I have the ratings right over here. Whole numbers, thank you. Thank you very much. [Audience chants "Four more yea

Création de la première row de la DB

In [25]:
import pandas as pd

df=pd.DataFrame(columns=["Date","Title","Categories","Nbr Sentences","Nbr Words", "Nbr Seconds", "Transcriptions_list"])
df.loc[len(df)]=[speech_date,speech_title,categories,nbr_sentences,nbr_words,nbr_seconds,trump_transcriptions]
df

,Date,Title,Categories,Nbr Sentences,Nbr Words,Nbr Seconds,Transcriptions_list
0,"October 26, 2020",Speech: Donald Trump Holds a Campaign Rally in...,"[Human Rights, Crime, Law and Justice, Electio...",1044,10182,3741,"[[00:00:00-00:00:28 (28 sec), Thank you very m..."


In [26]:
df.iloc[0].loc['Transcriptions_list']

[['00:00:00-00:00:28 (28 sec)',
  "Thank you very much, hello. Oh wow. We tried to come in here very low key. We wanted to make a low key appearance. Did anybody notice the helicopter flying a little low and real slow? Greatest pilots in the world. Greatest equipment in the world. That's what we have. Greatest equipment in the world."],
 ['00:00:28-00:00:50 (22 sec)',
  "But the greatest pilots. Nobody can do what our pilots can do, right, Mike? Hello Pennsylvania. Hello Pennsylvania. Eight days from now. Can you believe it? Eight days. Started off four years ago, remember our meetings and our -- it was a love fest, right from the beginning, wasn't it, and now it's more so."],
 ['00:00:50-00:01:27 (37 sec)',
  'You know what? It\'s more so than it was four years ago, and they\'re going to be finding that out very soon. They\'re going to be finding it out. You know, I have, I have the ratings right over here. Whole numbers, thank you. Thank you very much. [Audience chants "Four more yea